# Решение конечных игр методом итераций. Метод Брауна-Робинсон

In [2]:
import numpy as np
import pandas as pd

## Пропишим класс, который будет инициализировать игру с заданными параметрами (матрицой первого игрока и числом розыгрышей)
**Код запускается единожды для всех игр, поэтому не нужно вызывать его в каждой игре** 

In [ ]:
class BraunRobinson:
	"""
	Создаем Dataframe, в котороый будут записываться этапы разыгровок 
	"""
	def frame(self):
		dic = ['k','i']
		for bn in range(self.A.shape[1]):
			dic.append(f"B{bn+1}")
		dic.append('j')
		for an in range(self.A.shape[0]):
			dic.append(f"A{an+1}")
		dic.append('V_low',)
		dic.append('V_mid')
		dic.append('V_high')
		df = pd.DataFrame({"k": [i for i in range(1, self.n+1)]}, columns=dic)
		df = df.set_index('k')
		return df
	
	"""
	Функция, создающая экземпляр класса
	"""
	def __init__(self, A : np.ndarray, n : int):
		self.A = A
		self.n = n
		self.Dataframe = self.frame()
	
	"""
	Функция, записывающая итоги розыгрыша в Dataframe
	"""
	def fill(self, loc, dic) -> None:
		self.Dataframe.iloc[loc] = dic
	

	"""
	Функция, выполняющая алгоритм разыгрышей
	"""
	def Algorythm(self, i : int = None) -> dict:
		b = np.zeros(self.A.shape[1])
		a = np.zeros(self.A.shape[0])
		start_k = 0
		n = self.n
		
		"""
		Выполняем первую разыгровку отдельно, в случае, если задали стартовую стратегию пераого игрока
		"""
		if i is not None:
			# Выполняем разыгрыш (основной алгоритм)
			i-=1
			b += self.A[i]
			j = b.argmin()
			a += self.A[:,j]
			V_low = round(min(b), 2)
			V_up = round(max(a), 2)
			V_mid = round((V_low+V_up)/2, 2)
			
			# Формуируем словарь, в котором будут записываться данные для дальнейшей записи в Dataframe
			dic = {'i':i+1, 'j':j+1, 'V_low':V_low, 'V_mid':V_mid, 'V_high':V_up}
			for bn in range(self.A.shape[1]):
				dic[f"B{bn+1}"] = b[bn]
			for an in range(self.A.shape[0]):
				dic[f"A{an+1}"] = a[an]
			self.fill(loc=0, dic=dic)
		
			start_k = 1 # смещаем стартовую стратегию для запуска итерирования

		"""
		Итерационный процесс розыгрышей
		"""
		for k in range(start_k, n):

			# Выполняем разыгрыш (основной алгоритм)
			i = a.argmax()
			b += self.A[i]
			j = b.argmin()
			a += self.A[:,j]
			V_low = round(min(b)/(k+1), 2)
			V_up = round(max(a)/(k+1), 2)
			V_mid = round((V_low+V_up)/2, 2)

			# Формуируем словарь, в котором будут записываться данные для дальнейшей записи в Dataframe
			dic = {'i':i+1, 'j':j+1, 'V_low':V_low, 'V_mid':V_mid, 'V_high':V_up}
			for bn in range(self.A.shape[1]):
				dic[f"B{bn+1}"] = b[bn]
			for an in range(self.A.shape[0]):
				dic[f"A{an+1}"] = a[an]
			self.fill(loc=k, dic=dic)

		"""
		Формируем векторы частот стратегий для каждого игрока
		"""
		self.Dataframe['i'].astype(int)
		self.Dataframe['j'].astype(int)
		P = (self.Dataframe['i'].value_counts()/self.n).sort_index()
		Q = (self.Dataframe['j'].value_counts()/self.n).sort_index()
		return {'P':P, 'Q':Q, 'V_mid':V_mid}

## Задаем параметры игры 
- матрицу первого игрока - A
- количеством розыгрышей - n

In [ ]:
A = np.array(
	[
		[7,2,9],
		[2,9,0],
		[9,0,11]
	], 
).reshape(3,3) # Для написанного алгоритма важно соблюдать исходную размерность матрицы, для этого задаем ее данным оператором

n = 1000

print(f"A:\n{A}\n\n n: {n}")

A:
[[ 7  2  9]
 [ 2  9  0]
 [ 9  0 11]]

 n: 1000


## Создаем экземпляр класса игры с заданными параметрами и проводим разыгровку

In [5]:
gamemode = BraunRobinson(A, n) # Создаем игру с заданными параметрами (матрицей первого игрока - A, количеством розыгрышей - n)
game = gamemode.Algorythm(i=3) # Выполняем алгоритм розыгрышей с заданным стартовым параметром хода первого игрока (i=3) 


print(f"P: {game['P']}")
print(f"Q: {game['Q']}")
print(f"V_mid: {game['V_mid']}")
gamemode.Dataframe

P: i
1.0    0.251
2.0    0.500
3.0    0.249
Name: count, dtype: float64
Q: j
1.0    0.251
2.0    0.499
3.0    0.250
Name: count, dtype: float64
V_mid: 5.0


,i,B1,B2,B3,j,A1,A2,A3,V_low,V_mid,V_high
k,,,,,,,,,,,
1,3.0,9.0,0.0,11.0,2.0,2.0,9.0,0.0,0.0,4.5,9.0
2,2.0,11.0,9.0,11.0,2.0,4.0,18.0,0.0,4.5,6.75,9.0
3,2.0,13.0,18.0,11.0,3.0,13.0,18.0,11.0,3.67,4.84,6.0
4,2.0,15.0,27.0,11.0,3.0,22.0,18.0,22.0,2.75,4.12,5.5
5,1.0,22.0,29.0,20.0,3.0,31.0,18.0,33.0,4.0,5.3,6.6
...,...,...,...,...,...,...,...,...,...,...,...
996,2.0,4978.0,4982.0,4978.0,1.0,4980.0,4980.0,4980.0,5.0,5.0,5.0
997,1.0,4985.0,4984.0,4987.0,2.0,4982.0,4989.0,4980.0,5.0,5.0,5.0
998,2.0,4987.0,4993.0,4987.0,1.0,4989.0,4991.0,4989.0,5.0,5.0,5.0
